1. KLUE 데이터셋을 이용하여 CountVectorizer, TFIDF transformer를 이용해 DTM 2개 만들기 (각 데이터의 title의 value 1000개를 저장하여 문서로 사용)

from datasets import load_dataset
dataset = load_dataset("klue", "ynat")

3. 첫번째 문서 기준으로 코사인 유사도 계산
3. 가장 유사한 문서 10개의 내용 출력
- BOW
- TFIDF

In [6]:
from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
import numpy as np

dataset = load_dataset("klue", "ynat")

documents = dataset['train']['title'][:1000]

In [7]:
# CountVectorizer를 이용한 BOW DTM
cv = CountVectorizer()
bow_dtm = cv.fit_transform(documents)

# TfidfTransformer를 이용한 TF-IDF DTM
tfidf_transformer = TfidfTransformer()
tfidf_dtm = tfidf_transformer.fit_transform(bow_dtm)

In [8]:
# 코사인 유사도 계산
bow_features = bow_dtm.toarray()
tfidf_features = tfidf_dtm.toarray()

print(bow_features.shape)
print(tfidf_features.shape)

# 4장 실습 코드 기반의 코사인 유사도 계산 함수
def get_cosine_similarity(vec1, vec2):
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    # 분모가 0이 되는 오류(ZeroDivisionError) 방지
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return np.dot(vec1, vec2) / (norm1 * norm2)

# 기준이 되는 첫 번째 문서(index: 0) 벡터
target_idx = 0
target_bow = bow_features[target_idx]
target_tfidf = tfidf_features[target_idx]

bow_sim_scores = []
tfidf_sim_scores = []

# 첫 번째 문서와 나머지 전체 문서(1000개) 간의 유사도 순회 계산
for i in range(len(documents)):
    bow_sim = get_cosine_similarity(target_bow, bow_features[i])
    bow_sim_scores.append(bow_sim)
    
    tfidf_sim = get_cosine_similarity(target_tfidf, tfidf_features[i])
    tfidf_sim_scores.append(tfidf_sim)
    
# 정렬을 위해 numpy array로 변환
bow_sim_scores = np.array(bow_sim_scores)
tfidf_sim_scores = np.array(tfidf_sim_scores)

(1000, 4987)
(1000, 4987)


In [ ]:
# 4. 가장 유사한 문서 10개의 내용 출력 (-BOW, -TFIDF)

# 유사도가 높은 순으로 인덱스 정렬 (내림차순) 후 상위 10개 추출
top_10_bow_idx = np.argsort(bow_sim_scores)[::-1][:10]
top_10_tfidf_idx = np.argsort(tfidf_sim_scores)[::-1][:10]

print("BOW 기반 가장 유사한 10개")
for idx in top_10_bow_idx:
    print(documents[idx])

print("\nTF-IDF 기반 가장 유사한 문서 10개")
for idx in top_10_tfidf_idx:
    print(documents[idx])

BOW 기반 가장 유사한 10개
유튜브 내달 2일까지 크리에이터 지원 공간 운영
애플워치3 LTE 내달 15일 국내 출시
금융소비자원 DLS·DLF 피해자 상담센터 운영
한국투자 한국전력 내달 정책 불확실성 해소
유튜브 1분 넘는 광고도 소비자가 주목한다
사우디 이라크 스포츠도시 건설에 1조원 지원
유튜브 동영상 외에 음악감상용으로도 가장 많이 쓴다
신 넛크래커 극복 위해 도전적 연구 지원 강화
北매체 외부 지원 없어도 좋아…또 중국 겨냥
성신양회·GS글로벌 미얀마 시멘트부두 추진…해수부 지원

TF-IDF 기반 가장 유사한 문서 10개
유튜브 내달 2일까지 크리에이터 지원 공간 운영
금융소비자원 DLS·DLF 피해자 상담센터 운영
유튜브 1분 넘는 광고도 소비자가 주목한다
유튜브 동영상 외에 음악감상용으로도 가장 많이 쓴다
애플워치3 LTE 내달 15일 국내 출시
사우디 이라크 스포츠도시 건설에 1조원 지원
한국투자 한국전력 내달 정책 불확실성 해소
北매체 외부 지원 없어도 좋아…또 중국 겨냥
신 넛크래커 극복 위해 도전적 연구 지원 강화
野 전경련 어버이연합 뒷돈 지원 의혹 국회 진상조사


: 